In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [34]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 5, x.day-1, tzinfo=timezone)    
#     utc_to = datetime(x.year, x.month, x.day, tzinfo=timezone)
#     utc_to = datetime(x.year, x.month+1, 9, tzinfo=timezone)
    utc_to = datetime(x.year, x.month, x.day-2, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_M30, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
#     rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    
    rates_frame['sma'] = rates_frame['close'].rolling(window=300).mean()
    rates_frame['smaH'] = rates_frame['high'].rolling(window=30).mean()
#     rates_frame['smaL']= rates_frame['low'].rolling(window=20).mean()
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    return rates_frame

In [35]:
def get_rsi(close, lookback):
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     rsi_df = rsi_df.dropna()
    return rsi_df

# ibm['rsi_14'] = get_rsi(ibm['close'], 14)
# ibm = ibm.dropna()


In [36]:
def slope(x1, y1, x2, y2):
    return (y2-y1)/(x2-x1)

In [73]:
def go(a):
    g = []
    for i in range(0, len(a)):
        g.append(slope(0, a.iloc[i-1].rsi, 1, a.iloc[i].rsi))
    return g

In [106]:
symbol = "EURUSD"
a= get_values(symbol)
a['rsi'] = get_rsi(a['close'], 2)
a['rsi2'] = get_rsi(a['close'], 30)

# a['smaL']= a['rsi'].rolling(window=2).mean()
a['slope'] = go(a)
a = a.dropna()
# a = a[300:]
a

,open,close,sma,smaH,rsi,rsi2,slope
time,,,,,,,
2021-05-25 05:30:00,1.22262,1.22285,1.219444,1.221970,99.278526,60.355920,1.124382
2021-05-25 06:00:00,1.22285,1.22276,1.219473,1.222010,67.229216,59.763832,-32.049310
2021-05-25 06:30:00,1.22275,1.22266,1.219501,1.222061,39.146323,59.097461,-28.082892
2021-05-25 07:00:00,1.22266,1.22287,1.219528,1.222107,77.906884,60.064794,38.760561
2021-05-25 07:30:00,1.22286,1.22293,1.219556,1.222132,83.802341,60.342006,5.895457
...,...,...,...,...,...,...,...
2021-08-13 21:30:00,1.17973,1.17959,1.175475,1.176600,15.034431,71.911640,-12.716099
2021-08-13 22:00:00,1.17959,1.17973,1.175460,1.176788,55.665314,72.358693,40.630883
2021-08-13 22:30:00,1.17973,1.17981,1.175446,1.176985,71.332594,72.616331,15.667281


In [75]:
len(a)

2814

In [109]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000

for i in range(len(a)):
    if a.iloc[i-1].rsi >= 60.0 and a.iloc[i-2].rsi <= 40.0:
        if a.iloc[i].rsi < a.iloc[i-1].rsi:
            print(f"Slope--> {a.iloc[i-1].slope}||| {a.iloc[i-1].name} ||| Profit")
            index.append(a.iloc[i-1].slope)
            B.append(a.iloc[i-1].name)
            rsi1.append(round(a.iloc[i-1].rsi2, 3))
            if a.iloc[i-1].open > a.iloc[i-1].sma and a.iloc[i-1].close > a.iloc[i-1].sma:
                profit.append("UP")
            else:
                profit.append("Below")
        else:
            indexB.append(a.iloc[i-1].slope)
            profits.append(a.iloc[i-1].name)
            rsi2.append(round(a.iloc[i-1].rsi2, 3))
            print(f"Slope--> {a.iloc[i-1].slope}||| {a.iloc[i-1].name} ||| Loss")
            if a.iloc[i-1].open > a.iloc[i-1].sma and a.iloc[i-1].close > a.iloc[i-1].sma:
                p.append("UP")
            else:
                p.append("Below")
            
#         index.append((a.iloc[i-1].name, a.iloc[i-1].slope))
    
#     if a.iloc[i].rsi >= 65.0 and a.iloc[i-1].rsi <= 10.0 and a.iloc[i].slope > 37.0 and check == 0 :
#         buy_price = a.iloc[i].close
#         print("#"*20)
#         print(a.iloc[i].name)
#         print("*"*20)
#         check = 1
#         peck = a.iloc[i].rsi

#     elif check == 1:
#         sell_price = a.iloc[i].close
#         pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
#         print(f"{pp}---{round(a.iloc[i].rsi, 2)}-----{a.iloc[i].close}--{a.iloc[i].name}")


#         if a.iloc[i].rsi < a.iloc[i-1].rsi:
#             check = 0
#             profit.append(pp)

Slope--> 38.76056068819229||| 2021-05-25 07:00:00 ||| Loss
Slope--> 62.79351310202041||| 2021-05-25 09:00:00 ||| Loss
Slope--> 60.621465315874815||| 2021-05-25 14:00:00 ||| Profit
Slope--> 51.040730905883805||| 2021-05-25 17:00:00 ||| Profit
Slope--> 44.08945232834104||| 2021-05-25 19:30:00 ||| Loss
Slope--> 27.793613520693206||| 2021-05-25 23:30:00 ||| Loss
Slope--> 56.09186373746502||| 2021-05-26 03:00:00 ||| Profit
Slope--> 50.25540228121349||| 2021-05-26 05:00:00 ||| Loss
Slope--> 33.89515037346884||| 2021-05-26 12:00:00 ||| Loss
Slope--> 66.72681377628547||| 2021-05-26 21:30:00 ||| Loss
Slope--> 45.91931113734762||| 2021-05-27 00:30:00 ||| Loss
Slope--> 55.22258345417657||| 2021-05-27 08:30:00 ||| Profit
Slope--> 50.33586359148558||| 2021-05-27 09:30:00 ||| Loss
Slope--> 75.06045698826087||| 2021-05-27 16:30:00 ||| Loss
Slope--> 42.55769487773986||| 2021-05-27 20:00:00 ||| Loss
Slope--> 47.80355735658368||| 2021-05-28 09:30:00 ||| Profit
Slope--> 75.78721722273204||| 2021-05-31 00

Slope--> 80.68224060725012||| 2021-07-12 10:00:00 ||| Loss
Slope--> 55.55745006312122||| 2021-07-12 15:00:00 ||| Loss
Slope--> 55.98677807877732||| 2021-07-12 20:30:00 ||| Profit
Slope--> 53.01509163259665||| 2021-07-12 22:30:00 ||| Loss
Slope--> 44.70934917117913||| 2021-07-13 02:30:00 ||| Loss
Slope--> 42.74741286996698||| 2021-07-13 04:30:00 ||| Profit
Slope--> 45.1801803471148||| 2021-07-13 05:30:00 ||| Loss
Slope--> 59.34077548562631||| 2021-07-13 10:00:00 ||| Profit
Slope--> 66.98039505736976||| 2021-07-13 13:30:00 ||| Profit
Slope--> 52.29274859385623||| 2021-07-13 15:00:00 ||| Profit
Slope--> 40.981255037515616||| 2021-07-13 16:30:00 ||| Loss
Slope--> 43.842862144111756||| 2021-07-14 00:30:00 ||| Loss
Slope--> 36.1864266031788||| 2021-07-14 07:30:00 ||| Loss
Slope--> 63.72408438157143||| 2021-07-14 11:00:00 ||| Profit
Slope--> 37.273900425296205||| 2021-07-14 12:30:00 ||| Profit
Slope--> 54.82567987496661||| 2021-07-14 21:00:00 ||| Profit
Slope--> 64.98874345650302||| 2021-07-1

In [113]:
for i in range(len(indexB)):
    print(f"{round(indexB[i], 3)}    {rsi2[i]}    {p[i]}     {profits[i]}")


38.761    60.065    UP     2021-05-25 07:00:00
62.794    59.561    UP     2021-05-25 09:00:00
44.089    53.381    UP     2021-05-25 19:30:00
27.794    56.042    UP     2021-05-25 23:30:00
50.255    53.998    UP     2021-05-26 05:00:00
33.895    50.292    UP     2021-05-26 12:00:00
66.727    36.839    Below     2021-05-26 21:30:00
45.919    37.754    Below     2021-05-27 00:30:00
50.336    46.046    Below     2021-05-27 09:30:00
75.06    47.841    Below     2021-05-27 16:30:00
42.558    47.032    Below     2021-05-27 20:00:00
75.787    52.34    Below     2021-05-31 00:00:00
61.646    51.083    Below     2021-05-31 03:30:00
54.021    52.785    Below     2021-05-31 14:30:00
59.711    55.239    UP     2021-06-01 08:30:00
67.801    51.981    UP     2021-06-01 15:30:00
46.986    45.642    Below     2021-06-02 01:30:00
69.336    37.615    Below     2021-06-02 14:30:00
45.258    56.726    Below     2021-06-04 21:30:00
45.189    56.021    Below     2021-06-07 02:00:00
47.942    57.228    Below 

In [112]:
for i in range(len(index)):
    print(f"{round(index[i], 3)}    {rsi1[i]}   {profit[i]}    {B[i]}")
#     if index[i] > 70.0:
#         print(index[i], profit[i], B[i])

60.621    63.576   UP    2021-05-25 14:00:00
51.041    59.7   UP    2021-05-25 17:00:00
56.092    56.343   UP    2021-05-26 03:00:00
55.223    44.026   Below    2021-05-27 08:30:00
47.804    44.62   Below    2021-05-28 09:30:00
63.065    51.916   Below    2021-05-31 10:00:00
51.875    59.961   UP    2021-06-01 06:00:00
64.473    51.925   Below    2021-06-02 21:00:00
26.153    51.16   UP    2021-06-03 02:00:00
75.236    33.162   Below    2021-06-04 09:30:00
49.435    37.978   Below    2021-06-04 12:30:00
52.274    39.962   Below    2021-06-04 14:00:00
38.499    56.113   Below    2021-06-07 01:00:00
26.218    51.767   Below    2021-06-07 12:30:00
46.009    62.881   UP    2021-06-07 20:30:00
37.648    51.765   Below    2021-06-08 08:00:00
47.914    51.445   Below    2021-06-08 09:00:00
43.805    49.395   Below    2021-06-08 11:30:00
54.641    48.018   Below    2021-06-09 00:00:00
50.519    47.355   Below    2021-06-09 02:00:00
62.574    52.3   Below    2021-06-09 09:00:00
51.275    42.631

In [78]:
sum(profit)

0

In [66]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

Total negative sm -->-5.24
Total negative -->4
Total positive sm -->2.3200000000000003
Total positive -->4
Length 8


In [ ]:
# a['rsi'].plot(figsize=(15,6))
# rates_frame['ll'] = rates_frame['close'].rolling(window=100).mean()
a['slope'].plot(figsize=(15,6))

In [219]:
for i in range(len(a)):
    if a.iloc[i].open > a.iloc[i].sma and a.iloc[i].close > a.iloc[i].sma \
        and a.iloc[i].rsi <= 40.0 and a.iloc[i-2].rsi > 40.0:
        print(a.iloc[i].slope)
        print(a.iloc[i-3:i+15])
        break

-5.947101816687024
                        open    close       sma        rsi     slope
time                                                                
2021-05-19 20:00:00  1.22108  1.22099  1.214320  47.784633 -1.048973
2021-05-19 20:30:00  1.22099  1.22078  1.214333  46.909585 -0.622310
2021-05-19 21:00:00  1.22078  1.21903  1.214341  40.475541 -3.654546
2021-05-19 21:30:00  1.21902  1.21712  1.214342  35.015381 -5.947102
2021-05-19 22:00:00  1.21709  1.21756  1.214349  37.053058 -1.711241
2021-05-19 22:30:00  1.21756  1.21735  1.214350  36.484302  0.734460
2021-05-19 23:00:00  1.21735  1.21781  1.214352  38.633617  0.790279
2021-05-19 23:30:00  1.21781  1.21744  1.214357  37.568456  0.542077
2021-05-20 00:00:00  1.21744  1.21747  1.214358  37.713498 -0.460059
2021-05-20 00:30:00  1.21737  1.21754  1.214360  38.063235  0.247389
2021-05-20 01:00:00  1.21754  1.21740  1.214363  37.623125 -0.045187
2021-05-20 01:30:00  1.21740  1.21728  1.214366  37.238682 -0.412277
2021-05-20 02:0

In [210]:
pp = []
n = []
pt = []
nt = []
for i in range(len(a)):
#     if a.iloc[i].open < a.iloc[i].sma and a.iloc[i].close < a.iloc[i].sma and a.iloc[i].rsi < 40.0:
    if a.iloc[i].rsi < 40.0:

        if a.iloc[i].slope < -1.0:
            if a.iloc[i+1].slope > 0.0:
                pp.append(price_action(symbol, 0.02, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY))
#                 pt.append(1)
                
            elif a.iloc[i+1].slope < 0.0:
                print(a.iloc[i].name)
                n.append(price_action(symbol, 0.02, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY))
#                 nt.append(1)
#             pp = price_action(symbol, 0.02, a.iloc[i].open, a.iloc[i].open, mt5.ORDER_TYPE_SELL)

2021-05-18 03:00:00
2021-05-18 04:30:00
2021-05-21 11:30:00
2021-05-21 12:30:00
2021-05-21 13:00:00
2021-05-27 14:00:00
2021-05-27 17:30:00
2021-05-27 19:00:00
2021-05-27 19:30:00
2021-05-27 20:30:00
2021-05-27 21:00:00
2021-05-28 14:30:00
2021-06-03 11:30:00
2021-06-03 12:30:00
2021-06-04 09:30:00
2021-06-04 10:30:00
2021-06-04 11:00:00
2021-06-04 11:30:00
2021-06-04 14:30:00
2021-06-10 23:00:00
2021-06-11 13:30:00
2021-06-11 14:00:00
2021-06-11 17:30:00
2021-06-16 13:00:00
2021-06-16 21:30:00
2021-06-16 22:00:00
2021-06-17 04:00:00
2021-06-17 05:00:00
2021-06-17 06:30:00
2021-06-17 08:00:00
2021-06-17 09:30:00
2021-06-21 20:30:00
2021-06-21 23:30:00
2021-06-23 09:30:00
2021-07-02 21:00:00
2021-07-05 09:00:00
2021-07-06 03:00:00
2021-07-06 03:30:00
2021-07-06 11:00:00
2021-07-06 13:30:00
2021-07-06 14:00:00
2021-07-06 14:30:00
2021-07-09 16:30:00
2021-07-09 20:00:00
2021-07-09 21:30:00
2021-07-09 23:00:00
2021-07-13 18:30:00
2021-07-14 09:00:00
2021-07-14 10:30:00
2021-07-14 11:00:00


In [205]:
sum(pp)

91.64999999999999

In [206]:
sum(n)

-60.449999999999996

In [207]:
len(pp)

80

In [208]:
len(n)

74

In [209]:
sum(pp)+sum(n)

31.199999999999996

In [113]:
#less negatives more positives
#RSI EURUSD-25, M30

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if a.iloc[i].rsi >= 41.0:
        if a.iloc[i].open > a.iloc[i].sma and a.iloc[i].close > a.iloc[i].sma \
            and a.iloc[i].rsi <= 40.0 and a.iloc[i-2].rsi > 40.0 \
            and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
#             print("RSI   SLOPE")
#             print(f"{round(a.iloc[i].rsi,2)}   {round(a.iloc[i].slope,2)}")
#             print("SMA   SMAH")
#             print(f"{peck}   {up}")
# #             print(f"{round(a.iloc[i].sma,6)}   {round(a.iloc[i].smaH,6)}")
#             print("CLOSE")
#             print(f"{a.iloc[i].Close}")
#             print(a.iloc[i].smaL)
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(f"{pp}---{round(a.iloc[i].rsi, 2)}--{round(a.iloc[i].slope, 2)}---{a.iloc[i].close}--{a.iloc[i].name}")
#             profit.append(pp)
#             check = 0
            
            
#             if pp > 0.0:
#                 profit.append(pp)
#                 check = 0
#             if pp < -3.0:
#                 profit.append(pp)
#                 check = 0
#             if a.iloc[i].rsi >= 49.0:
#                 if a.iloc[i].smaL >= a.iloc[i-1].smaL:
#                     pass
#                 else:
#                     profit.append(pp)
#                     check = 0
#             if a.iloc[i].rsi - a.iloc[i-1].rsi >= 2.0 :
#                 profit.append(pp)
#                 check = 0
            if a.iloc[i].rsi >= 50.0:
                profit.append(pp)
                check = 0
#             if a.iloc[i].rsi <= 40.0:
#                 profit.append(pp)
#                 check = 0

####################
2012-03-06 00:00:00
********************


AttributeError: 'Series' object has no attribute 'slope'